# 🕸️ Delhivery Logistics Network — Graph Construction
### Notebook 3 

**Objective:** Transform the clean trip dataset into a directed weighted graph.  
Facilities become nodes. Corridors become edges.  
Edge weights capture the median delay ratio per corridor.

**This graph is the foundation of everything that follows:**
- Bottleneck analysis (Notebook 4)
- Graph-enhanced ETA prediction (Notebook 6)
- FTL vs Carting framework (Notebook 7)

---

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import plotly.graph_objects as go
import os
import warnings

warnings.filterwarnings('ignore')

os.makedirs('../outputs/visualisations', exist_ok=True)
os.makedirs('../outputs/model_results', exist_ok=True)

print("✅ Libraries loaded")
print(f"   pandas    : {pd.__version__}")
print(f"   numpy     : {np.__version__}")
print(f"   networkx  : {nx.__version__}")

## 📂 Step 1 — Load Clean Dataset

In [ ]:
# Load clean dataset
df = pd.read_csv('../data/delivery_data_clean.csv')

# Fix datetime columns
df['od_start_time'] = pd.to_datetime(df['od_start_time'])
df['od_end_time']   = pd.to_datetime(df['od_end_time'])

print("=" * 55)
print("       CLEAN DATASET LOADED")
print("=" * 55)
print(f"\n  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]}")
print(f"\n  Unique source hubs      : {df['source_center'].nunique():,}")
print(f"  Unique destination hubs : {df['destination_center'].nunique():,}")
print(f"  Unique corridors        : {df['corridor_key'].nunique():,}")
print(f"  Route types             : {df['route_type'].unique()}")
print("\n" + "=" * 55)

## 🔧 Step 2 — Compute Edge Weights

Each corridor (source → destination) becomes a graph edge.  
The edge weight is the **median delay ratio** for that corridor.  

**Why median and not mean?**  
Median is more robust to the remaining outliers.  
A single extremely delayed trip won't distort the edge weight.  
This gives us a more reliable signal of corridor-level delay.

We also stratify by route type — FTL and Carting corridors  
have different delay profiles and will be analyzed separately.

In [ ]:
# Compute corridor-level edge weights
corridor_stats = df.groupby(
    ['source_center', 'destination_center', 'corridor_key', 'route_type']
).agg(
    trip_count          = ('delay_ratio_clean', 'count'),
    median_delay        = ('delay_ratio_clean', 'median'),
    mean_delay          = ('delay_ratio_clean', 'mean'),
    std_delay           = ('delay_ratio_clean', 'std'),
    median_actual_time  = ('actual_time', 'median'),
    median_osrm_time    = ('osrm_time', 'median'),
    median_distance     = ('actual_distance_to_destination', 'median'),
    chronic_rate        = ('delay_ratio_clean', lambda x: (x > 1.2).mean() * 100)
).reset_index()

print(f"Corridor Statistics Computed:\n")
print(f"  Total corridor-route combinations : {len(corridor_stats):,}")
print(f"  FTL corridors                     : {(corridor_stats['route_type']=='FTL').sum():,}")
print(f"  Carting corridors                 : {(corridor_stats['route_type']=='Carting').sum():,}")
print(f"\nDelay Ratio Statistics Across All Corridors:")
print(f"  Min median delay  : {corridor_stats['median_delay'].min():.4f}")
print(f"  Mean median delay : {corridor_stats['median_delay'].mean():.4f}")
print(f"  Max median delay  : {corridor_stats['median_delay'].max():.4f}")
print(f"\nTop 5 most delayed corridors:")
top5 = corridor_stats.nlargest(5, 'median_delay')[
    ['source_center', 'destination_center', 'route_type', 'median_delay', 'trip_count']
]
print(top5.to_string(index=False))

## 🕸️ Step 3 — Build the Full Directed Graph

We build a directed weighted graph where:
- **Nodes** = unique facility/hub codes
- **Edges** = corridors between facilities
- **Edge weight** = median delay ratio for that corridor

Node attributes store hub-level statistics.  
Edge attributes store corridor-level statistics.

In [ ]:
# Build directed graph
G = nx.DiGraph()

# Add all unique hubs as nodes
all_hubs = pd.concat([
    df['source_center'],
    df['destination_center']
]).unique()

G.add_nodes_from(all_hubs)

print(f"Nodes added: {G.number_of_nodes():,}")

# Add node attributes
hub_stats = df.groupby('source_center').agg(
    outbound_trips    = ('trip_uuid', 'count'),
    avg_delay_out     = ('delay_ratio_clean', 'mean'),
    states_served     = ('dest_state', 'nunique')
).reset_index()

for _, row in hub_stats.iterrows():
    if G.has_node(row['source_center']):
        G.nodes[row['source_center']]['outbound_trips'] = row['outbound_trips']
        G.nodes[row['source_center']]['avg_delay_out']  = row['avg_delay_out']
        G.nodes[row['source_center']]['states_served']  = row['states_served']

print(f"Node attributes added")

In [ ]:
# Add edges with attributes
# Use overall corridor stats (all route types combined)
corridor_overall = df.groupby(
    ['source_center', 'destination_center', 'corridor_key']
).agg(
    trip_count         = ('delay_ratio_clean', 'count'),
    median_delay       = ('delay_ratio_clean', 'median'),
    mean_delay         = ('delay_ratio_clean', 'mean'),
    median_actual_time = ('actual_time', 'median'),
    median_osrm_time   = ('osrm_time', 'median'),
    median_distance    = ('actual_distance_to_destination', 'median'),
    chronic_rate       = ('delay_ratio_clean', lambda x: (x > 1.2).mean() * 100),
    ftl_pct            = ('route_type', lambda x: (x == 'FTL').mean() * 100)
).reset_index()

# Add edges
for _, row in corridor_overall.iterrows():
    G.add_edge(
        row['source_center'],
        row['destination_center'],
        weight             = row['median_delay'],
        trip_count         = row['trip_count'],
        median_delay       = row['median_delay'],
        mean_delay         = row['mean_delay'],
        median_actual_time = row['median_actual_time'],
        median_osrm_time   = row['median_osrm_time'],
        median_distance    = row['median_distance'],
        chronic_rate       = row['chronic_rate'],
        ftl_pct            = row['ftl_pct'],
        corridor_key       = row['corridor_key']
    )

print(f"Edges added: {G.number_of_edges():,}")
print(f"\nGraph Summary:")
print(f"  Nodes (hubs)     : {G.number_of_nodes():,}")
print(f"  Edges (corridors): {G.number_of_edges():,}")
print(f"  Is directed      : {G.is_directed()}")
print(f"  Is weighted      : True (edge weight = median delay ratio)")

## 📊 Step 4 — Graph Basic Statistics

Before any analysis, we understand the structural properties  
of the graph. These stats tell us how complex and connected  
the logistics network actually is.

In [ ]:
# Basic graph statistics
print("=" * 55)
print("         GRAPH STRUCTURAL STATISTICS")
print("=" * 55)

# Density
density = nx.density(G)

# Degree statistics
in_degrees  = dict(G.in_degree())
out_degrees = dict(G.out_degree())

avg_in  = np.mean(list(in_degrees.values()))
avg_out = np.mean(list(out_degrees.values()))
max_in  = max(in_degrees.values())
max_out = max(out_degrees.values())

# Weakly connected components
wcc = list(nx.weakly_connected_components(G))

print(f"""
  Nodes                    : {G.number_of_nodes():,}
  Edges                    : {G.number_of_edges():,}
  Graph Density            : {density:.6f}
  
  In-degree  avg           : {avg_in:.2f}
  In-degree  max           : {max_in}
  Out-degree avg           : {avg_out:.2f}
  Out-degree max           : {max_out}
  
  Weakly connected components : {len(wcc)}
  Largest component size      : {max(len(c) for c in wcc):,} nodes
  Isolated nodes              : {sum(1 for c in wcc if len(c) == 1)}
""")
print("=" * 55)

# Most connected hub
most_connected = max(dict(G.degree()).items(), key=lambda x: x[1])
print(f"\n  Most connected hub: {most_connected[0]}")
print(f"  Total connections : {most_connected[1]}")

## 🔴 Step 5 — Identify Chronic Delay Corridors

A corridor is "chronically delayed" if its median delay ratio > 1.2  
meaning actual time consistently exceeds OSRM by more than 20%.  
These are the red edges in our network visualization.

In [ ]:
# Identify chronic delay edges
chronic_edges = [
    (u, v, d) for u, v, d in G.edges(data=True)
    if d.get('median_delay', 0) > 1.2
]

normal_edges = [
    (u, v, d) for u, v, d in G.edges(data=True)
    if d.get('median_delay', 0) <= 1.2
]

print(f"Chronic Delay Corridor Analysis:\n")
print(f"  Total corridors          : {G.number_of_edges():,}")
print(f"  Chronic corridors (>1.2) : {len(chronic_edges):,} ({len(chronic_edges)/G.number_of_edges()*100:.1f}%)")
print(f"  Normal corridors (≤1.2)  : {len(normal_edges):,} ({len(normal_edges)/G.number_of_edges()*100:.1f}%)")

print(f"\nTop 10 Most Chronically Delayed Corridors:")
print("-" * 75)

chronic_sorted = sorted(chronic_edges, key=lambda x: x[2]['median_delay'], reverse=True)
for u, v, d in chronic_sorted[:10]:
    print(f"  {u[:20]:<20} → {v[:20]:<20} | "
          f"Delay: {d['median_delay']:.3f} | "
          f"Trips: {d['trip_count']:>4} | "
          f"Chronic rate: {d['chronic_rate']:.1f}%")
print("-" * 75)

## 📊 Step 6 — Degree Distribution

The degree distribution tells us if this network follows  
a power law (few hubs with very high connectivity — typical  
of real logistics networks) or is more uniform.  
High-degree hubs are bottleneck candidates.

In [ ]:
# Degree distribution
degrees = [d for n, d in G.degree()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(degrees, bins=40, color='#4A90D9',
             edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Degree (in + out connections)', fontsize=12)
axes[0].set_ylabel('Number of Hubs', fontsize=12)
axes[0].set_title('Hub Degree Distribution', fontsize=13, fontweight='bold')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Top 20 hubs by degree
top_degree = sorted(dict(G.degree()).items(),
                    key=lambda x: x[1], reverse=True)[:20]
hubs, degs = zip(*top_degree)

short_labels = [h[:15] for h in hubs]
colors = ['#E05C5C' if d == max(degs) else '#4A90D9' for d in degs]

axes[1].barh(short_labels[::-1], degs[::-1],
             color=colors[::-1], edgecolor='white', height=0.6)
axes[1].set_xlabel('Total Degree', fontsize=12)
axes[1].set_title('Top 20 Hubs by Connectivity', fontsize=13, fontweight='bold')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].tick_params(axis='y', labelsize=8)

plt.suptitle('Network Degree Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/visualisations/degree_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")

## 🎨 Step 7 — Network Visualization

We visualize the graph with:
- **Node size** = total degree (more connections = bigger node)
- **Node color** = average outbound delay (red = high delay)
- **Edge color** = median delay ratio (red = chronic, green = normal)

Note: For large graphs we sample the most connected subgraph  
to keep the visualization readable.

In [ ]:
# Get top connected nodes for visualization
top_nodes = sorted(dict(G.degree()).items(),
                   key=lambda x: x[1], reverse=True)[:80]
top_node_ids = [n for n, d in top_nodes]

# Create subgraph
subG = G.subgraph(top_node_ids)

print(f"Visualization subgraph:")
print(f"  Nodes : {subG.number_of_nodes()}")
print(f"  Edges : {subG.number_of_edges()}")

# Layout
pos = nx.spring_layout(subG, k=2, seed=42)

# Node sizes based on degree
node_degrees = dict(subG.degree())
node_sizes = [node_degrees[n] * 50 + 100 for n in subG.nodes()]

# Node colors based on avg delay
node_delays = []
for n in subG.nodes():
    delay = G.nodes[n].get('avg_delay_out', 1.0)
    node_delays.append(delay if not np.isnan(delay) else 1.0)

# Edge colors based on delay ratio
edge_colors = []
edge_widths = []
for u, v, d in subG.edges(data=True):
    delay = d.get('median_delay', 1.0)
    if delay > 1.5:
        edge_colors.append('#E05C5C')
        edge_widths.append(2.0)
    elif delay > 1.2:
        edge_colors.append('#E8A838')
        edge_widths.append(1.5)
    else:
        edge_colors.append('#7BC8A4')
        edge_widths.append(0.8)

fig, ax = plt.subplots(figsize=(16, 12))

# Draw edges
nx.draw_networkx_edges(
    subG, pos, ax=ax,
    edge_color=edge_colors,
    width=edge_widths,
    alpha=0.7,
    arrows=True,
    arrowsize=8,
    connectionstyle='arc3,rad=0.1'
)

# Draw nodes
sc = nx.draw_networkx_nodes(
    subG, pos, ax=ax,
    node_size=node_sizes,
    node_color=node_delays,
    cmap=plt.cm.RdYlGn_r,
    vmin=1.0, vmax=3.0,
    alpha=0.9
)

# Labels for top 20 nodes only
top20_ids = set([n for n, d in top_nodes[:20]])
labels = {n: n[:8] for n in subG.nodes() if n in top20_ids}
nx.draw_networkx_labels(
    subG, pos, labels, ax=ax,
    font_size=6, font_color='black', font_weight='bold'
)

plt.colorbar(sc, ax=ax, label='Avg Outbound Delay Ratio', shrink=0.6)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='#E05C5C', linewidth=2, label='Chronic delay (>1.5)'),
    Line2D([0], [0], color='#E8A838', linewidth=2, label='Moderate delay (1.2-1.5)'),
    Line2D([0], [0], color='#7BC8A4', linewidth=2, label='Normal delay (<1.2)')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10)

ax.set_title(
    f'Delhivery Logistics Network\n'
    f'Top 80 Hubs | {subG.number_of_edges()} Corridors Shown',
    fontsize=14, fontweight='bold', pad=20
)
ax.axis('off')

plt.tight_layout()
plt.savefig('../outputs/visualisations/network_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Network graph saved")

## 💾 Step 8 — Save Graph Object

We save the graph in two formats:
- **GraphML** — standard format, loadable by any graph tool
- **Edge list CSV** — easy to inspect and load in pandas

In [ ]:
# Save graph as GraphML
nx.write_graphml(G, '../outputs/model_results/logistics_graph.graphml')
print("✅ Graph saved as GraphML")

# Save edge list as CSV
edges_data = []
for u, v, d in G.edges(data=True):
    row = {'source': u, 'destination': v}
    row.update(d)
    edges_data.append(row)

edges_df = pd.DataFrame(edges_data)
edges_df.to_csv('../outputs/model_results/corridor_edge_list.csv', index=False)
print(f"✅ Edge list saved as CSV")
print(f"   Shape: {edges_df.shape}")

# Save node list as CSV
nodes_data = []
for n, d in G.nodes(data=True):
    row = {'hub_id': n}
    row.update(d)
    nodes_data.append(row)

nodes_df = pd.DataFrame(nodes_data)
nodes_df.to_csv('../outputs/model_results/hub_node_list.csv', index=False)
print(f"✅ Node list saved as CSV")
print(f"   Shape: {nodes_df.shape}")

In [ ]:
# Final graph summary
print("=" * 60)
print("         GRAPH CONSTRUCTION COMPLETE")
print("=" * 60)
print(f"""
GRAPH STRUCTURE
  Total nodes (hubs)       : {G.number_of_nodes():,}
  Total edges (corridors)  : {G.number_of_edges():,}
  Graph density            : {nx.density(G):.6f}
  Directed                 : Yes

DELAY PROFILE
  Chronic corridors (>1.2) : {len(chronic_edges):,} ({len(chronic_edges)/G.number_of_edges()*100:.1f}%)
  Normal corridors (≤1.2)  : {len(normal_edges):,} ({len(normal_edges)/G.number_of_edges()*100:.1f}%)
  Mean edge weight         : {np.mean([d['median_delay'] for u,v,d in G.edges(data=True)]):.4f}

FILES SAVED
  logistics_graph.graphml  → full graph object
  corridor_edge_list.csv   → edge attributes
  hub_node_list.csv        → node attributes
""")
print("=" * 60)
print("  NEXT STEP → 04_bottleneck_analysis.ipynb")
print("=" * 60)

---
## ✅ Graph Construction Complete

### What we built:
- A **directed weighted graph** of Delhivery's logistics network
- Every hub is a node with degree and delay attributes
- Every corridor is an edge with delay ratio and trip count
- Chronic delay corridors identified and flagged

### Graph saved to:
- `outputs/model_results/logistics_graph.graphml`
- `outputs/model_results/corridor_edge_list.csv`
- `outputs/model_results/hub_node_list.csv`

---
### ➡️ Next: `04_bottleneck_analysis.ipynb`